# LinguoMT — Central Experiment Runner

This notebook runs all four speech-translation evaluation pipelines, consolidates the
results, and produces a structured **Markdown results report** to support paper writing.

---

## What you will get

After a full run you receive a downloadable ZIP containing:

| File / Folder | Contents |
|---|---|
| `results_report.md` | Analysis report: tables, cross-model comparison, SOTA gap, discussion |
| `consolidated_metrics/` | Merged CSVs across all experiments (`text`, `asr`, `audio`) |
| `<experiment>/tables/` | Per-experiment Markdown and CSV summary tables |
| `<experiment>/plots/` | Score charts and EDA visualisations (PNG) |
| `<experiment>/interpretations/` | Auto-generated text interpretation fragments |
| `<experiment>/summaries/` | Per-language and per-experiment summary Markdown |

---

## The 5 papers and the `PAPER_MODE` values

Each run of this notebook targets **one** of the five papers.
Set `PAPER_MODE` in Step 4 to tell the framework which paper you are running.

| `PAPER_MODE` | Paper | What it measures | Key output |
|---|---|---|---|
| `"benchmark"` | **Paper 1 — LinguoMT** | Zero-shot baseline quality across models, languages, datasets, and translation directions | Baseline BLEU/ChrF/WER tables + SOTA gap analysis |
| `"adaptation"` | **Paper 2 — LinguoMT-Adapt** | How much LoRA fine-tuning improves over the pretrained baseline | Before/after comparison tables + data scaling curves |
| `"audio"` | **Paper 3 — LinguoMT-Audio** | Impact of audio preprocessing strategies on translation quality | Strategy comparison: direct / normalised / trimmed / chunk-based |
| `"cascade"` | **Paper 4 — LinguoMT-Cascade** | Cascade (ASR+MT) vs end-to-end architecture tradeoffs | Oracle ceiling, error-propagation curves, latency/VRAM comparison |
| `"transfer"` | **Paper 5 — LinguoMT-Transfer** | Cross-lingual transfer across Niger-Congo vs Afro-Asiatic languages | Typological similarity, transfer efficiency, few-shot scaling |

---

## The 4 experiments

Each experiment pairs a **model** with a **dataset**. Together they cover all five papers.

| # | `EXPERIMENT` key | Model | Dataset | Languages | Needed for |
|---|---|---|---|---|---|
| 1 | `AfricanCeltic__SeamlessM4Tv2` | SeamlessM4T-v2-Large | African-Celtic / IWSLT 2026 | Igbo · Yoruba | Papers 1–5 |
| 2 | `AfricanCeltic__WhisperNLLB` | Whisper-large-v3 + NLLB-600M | African-Celtic / IWSLT 2026 | Yoruba · Hausa | Papers 1, 4 |
| 3 | `FLEURS__SeamlessM4Tv2` | SeamlessM4T-v2-Large | FLEURS (google/fleurs) | Igbo · Yoruba · Swahili | Papers 1–5 |
| 4 | `FLEURS__WhisperNLLB` | Whisper-large-v3 + NLLB-600M | FLEURS (google/fleurs) | Yoruba · Hausa · Swahili | Papers 1–5 |

---

## Language coverage matrix

Not every language is available in every experiment — model or dataset constraints apply.

| Language | ISO | Family | AC×Seamless | AC×Whisper | FLEURS×Seamless | FLEURS×Whisper | Reason for any gap |
|---|---|---|:---:|:---:|:---:|:---:|---|
| Igbo | ibo | Niger-Congo | ✓ | — | ✓ | — | No Whisper language token for Igbo |
| Yoruba | yor | Niger-Congo | ✓ | ✓ | ✓ | ✓ | Fully supported across all experiments |
| Hausa | hau | Afro-Asiatic | — | ✓ | — | ✓ | African-Celtic has no Hausa audio; SeamlessM4T `hau` not in S2TT list |
| Swahili | swh | Bantu | — | — | ✓ | ✓ | FLEURS only; substitutes Hausa for 3-language parity |

---

## Estimated runtimes (T4 GPU)

| Mode | Per experiment | 4 experiments total | Use when |
|------|---------------|---------------------|----------|
| `DEBUG_MODE = True` | ~4 min | ~15 min | Smoke-testing the pipeline; verifying setup is correct |
| `DEBUG_MODE = False` | ~40–60 min | ~2.5–3 h | Full paper run on publication-quality data |

> **Runtime requirement:** GPU is recommended for Steps 7–10.
> Go to **Runtime → Change runtime type → T4 GPU**.
> Steps 1–6 (setup, download, data prep) run fine on CPU.

---

## Quick start

1. **Set GPU** → Runtime → Change runtime type → T4 GPU
2. **Run Steps 1–3** (Drive, clone, deps) — these never change
3. **Run Step 4 once** — set cache paths and `MAX_CACHED_PAIRS`
4. **Run Step 5 once ever** — downloads models to Drive (~25–35 min first run, < 1 min after)
5. **Run Step 6 once per config** — pre-builds sample pairs (~5–15 min first time, < 5 sec after)
6. **Edit Step 7** — set `PAPER_MODE`, `DEBUG_MODE`, eval parameters (change and re-run freely)
7. **Run Steps 8–11** — experiments; change Step 7 settings and repeat only Steps 8–11


## Models, Datasets & Language Support

---

### Models

| Role | Model | HuggingFace |
|---|---|---|
| End-to-end speech-to-text translation | SeamlessM4T-v2-Large | [facebook/seamless-m4t-v2-large](https://huggingface.co/facebook/seamless-m4t-v2-large) |
| ASR — cascade pipeline | Whisper large-v3 | [openai/whisper-large-v3](https://huggingface.co/openai/whisper-large-v3) |
| Machine translation — cascade pipeline | NLLB-200-distilled-600M | [facebook/nllb-200-distilled-600M](https://huggingface.co/facebook/nllb-200-distilled-600M) |

### Datasets

| Dataset | Description | HuggingFace |
|---|---|---|
| African-Celtic | IWSLT 2026 African-Celtic dataset — Igbo, Yoruba, Hausa speech + English translations | [McGill-NLP/african_celtic_dataset](https://huggingface.co/datasets/McGill-NLP/african_celtic_dataset) |
| FLEURS | Google FLEURS benchmark — 102 languages, clean read-speech utterances | [google/fleurs](https://huggingface.co/datasets/google/fleurs) |

---

### Full language support matrix

The framework registers 16 African languages. The ✓ below means the language is supported
by **both** the model and the dataset for that experiment — you can add it to `MANUAL_LANGUAGES` today.

| Language | ISO | Family | Region | AC × SeamlessM4T | AC × Whisper+NLLB | FLEURS × SeamlessM4T | FLEURS × Whisper+NLLB |
|---|---|---|---|:---:|:---:|:---:|:---:|
| Igbo | ibo | Niger-Congo | Nigeria | ✓ | — | ✓ | — |
| Yoruba | yor | Niger-Congo | Nigeria | ✓ | ✓ | ✓ | ✓ |
| Hausa | hau | Afro-Asiatic | West Africa | — | ✓ | — | ✓ |
| Swahili | swh | Bantu | East Africa | — | — | ✓ | ✓ |
| Amharic | amh | Semitic | Ethiopia | — | — | ✓ | — |
| Wolof | wol | Niger-Congo | Senegal | — | — | ✓ | — |
| Somali | som | Cushitic | Horn of Africa | — | — | ✓ | — |
| Oromo | orm | Cushitic | Ethiopia / Kenya | — | — | ✓ | — |
| Zulu | zul | Bantu | South Africa | — | — | ✓ | — |
| Xhosa | xho | Bantu | South Africa | — | — | ✓ | — |
| Lingala | lin | Bantu | DRC / Congo | — | — | ✓ | — |
| Fula | ful | Niger-Congo | West Africa | — | — | ✓ | — |
| Twi | twi | Niger-Congo | Ghana | — | — | ✓ | — |
| Ewe | ewe | Niger-Congo | Ghana / Togo | — | — | ✓ | — |
| Kikuyu | kik | Bantu | Kenya | — | — | ✓ | — |
| Luo | luo | Nilo-Saharan | Kenya | — | — | — | — |

**Why each gap exists:**

| Language | Excluded from | Reason |
|---|---|---|
| Igbo | All Whisper experiments | Whisper has no language token for Igbo |
| Hausa | FLEURS × SeamlessM4T | `hau` is absent from SeamlessM4T's S2TT output vocabulary |
| Swahili | African-Celtic experiments | African-Celtic / IWSLT 2026 has no Swahili audio |
| Amharic – Kikuyu | All Whisper experiments | None of these languages have a Whisper language token |
| Luo | All experiments | No standalone FLEURS config; not in African-Celtic |

---

### Extending experiments — how to add new languages

Only **FLEURS × SeamlessM4T-v2** allows new languages without any code changes —
14 languages are available there (all except Hausa and Luo).
The other experiments are constrained by the dataset (African-Celtic has only 3 languages)
or by Whisper's limited African language token support (only Yoruba, Hausa, Swahili).

**To add languages to FLEURS × SeamlessM4T-v2:**

1. Open `FLEURS__SeamlessM4Tv2/notebooks/run_experiment.py`
2. Edit `MANUAL_LANGUAGES` — use the lowercase key from the `Language` column above:

```python
# Current baseline (3 languages)
MANUAL_LANGUAGES = ["igbo", "yoruba", "swahili"]

# Extended example — add 3 more
MANUAL_LANGUAGES = ["igbo", "yoruba", "swahili", "amharic", "wolof", "somali"]

# All 14 available languages at once
MANUAL_LANGUAGES = [
    "igbo", "yoruba", "swahili",
    "amharic", "wolof", "somali", "oromo",
    "zulu", "xhosa", "lingala", "fula",
    "twi", "ewe", "kikuyu",
]

# Auto-detect from model + dataset capabilities (less reproducible)
MANUAL_LANGUAGES = None
```

**Estimated FULL runtime when adding languages (FLEURS × SeamlessM4T-v2):**

| Languages | Approx. runtime |
|---|---|
| 3 — current baseline | ~45 min |
| 6 | ~90 min |
| 10 | ~2.5 h |
| 14 — all available | ~3.5 h |


## Strategy Guide — Read This Before Editing Step 4

This section explains every configuration decision, what each experiment does,
and what results to expect. After reading this you will know exactly what to set.

---

## Experiment descriptions

### Experiment 1 — AfricanCeltic × SeamlessM4T-v2

**Model:** `facebook/seamless-m4t-v2-large` — a massively multilingual **end-to-end**
model that performs speech-to-text translation (S2TT), ASR, and text translation in
a single pass with no intermediate transcription step.

**Dataset:** `McGill-NLP/african_celtic_dataset` — the IWSLT 2026 African-Celtic dataset.
Speech utterances in African languages paired with English translations.

**Languages evaluated:** Igbo (ibo) · Yoruba (yor)
*(Hausa excluded because this dataset contains no Hausa audio.)*

**What it tests:** How well a pretrained end-to-end multilingual model performs
zero-shot on Nigerian languages from the Niger-Congo family. Establishes the S2TT
quality ceiling on the African-Celtic domain.

**Expected results:** BLEU scores of roughly 5–20 depending on language; Yoruba
tends to outperform Igbo because Yoruba has more representation in the model's
training data. ASR WER is typically 30–60% for these low-resource languages.

---

### Experiment 2 — AfricanCeltic × Whisper+NLLB

**Models:**
- `openai/whisper-large-v3` — state-of-the-art multilingual ASR; transcribes speech to text
- `facebook/nllb-200-distilled-600M` — 200-language MT model; translates the transcript

**Architecture:** **Cascade** — the two models are chained: Whisper produces a transcript
in the source language, NLLB translates it to English. Errors in the ASR step
propagate into the MT step (error propagation).

**Dataset:** `McGill-NLP/african_celtic_dataset`

**Languages evaluated:** Yoruba (yor) · Hausa (hau)
*(Igbo excluded: Whisper does not have a language token for Igbo — it cannot be
forced to transcribe in Igbo.)*

**What it tests:** How a cascade architecture compares to end-to-end on the same
domain. Also establishes the ASR quality (WER) for Yoruba and Hausa.

**Expected results:** Cascade BLEU tends to be lower than end-to-end for high-WER
languages, but can match or exceed end-to-end when ASR quality is good (WER < 30%).

---

### Experiment 3 — FLEURS × SeamlessM4T-v2

**Model:** `facebook/seamless-m4t-v2-large` (same as Experiment 1)

**Dataset:** `google/fleurs` — a large multilingual speech benchmark with clean,
read-speech utterances from Common Voice. More diverse in speaking style and
recording conditions than African-Celtic.

**Languages evaluated:** Igbo (ibo) · Yoruba (yor) · Swahili (swh) · Amharic (amh) · Zulu (zul) · Wolof (wol)

**What it tests:** Zero-shot S2TT quality on the FLEURS benchmark, which is the
standard evaluation set used by most published multilingual speech translation systems.
Results here are directly comparable to published SOTA numbers.

**Expected results:** Slightly higher BLEU than African-Celtic because FLEURS uses
cleaner recordings. Swahili (Bantu, widely represented in training data) typically
outperforms Igbo and Yoruba.

---

### Experiment 4 — FLEURS × Whisper+NLLB

**Models:** `openai/whisper-large-v3` + `facebook/nllb-200-distilled-600M` (cascade)

**Dataset:** `google/fleurs`

**Languages evaluated:** Yoruba (yor) · Hausa (hau) · Swahili (swh)
*(Igbo excluded: no Whisper language token. Swahili substitutes for 3-language parity.)*

**What it tests:** Cascade performance on the standard FLEURS benchmark.
Provides the cascade baseline used in Papers 1, 3, 4, and 5.

**Expected results:** Swahili cascade results are usually strong (Swahili has good
Whisper ASR). Hausa varies more due to dialect diversity in the recordings.

---

## Per-paper configuration recipes

### Paper 1 — Benchmark  (`PAPER_MODE = "benchmark"`)

**Research question:** How well do pretrained SeamlessM4T-v2 and Whisper+NLLB perform
on African languages **without any fine-tuning**? How do they compare to published SOTA?

**Setup:**
```python
PAPER_MODE        = "benchmark"
DEBUG_MODE        = False          # False for publication-quality results
ENABLE_FINETUNING = False          # No fine-tuning — zero-shot evaluation only
SCALING_BUDGETS   = []             # Not applicable
SOTA_FILE         = "sota/paper1_benchmark/sota_results.csv"  # or "" if not yet available
EXPERIMENTS       = ["AfricanCeltic__SeamlessM4Tv2", "AfricanCeltic__WhisperNLLB",
                     "FLEURS__SeamlessM4Tv2",        "FLEURS__WhisperNLLB"]
```

**Expected outputs:**
- BLEU and ChrF for Source→English and English→Source directions, per language per experiment
- WER and CER per language (from the ASR component)
- Cross-model table: SeamlessM4T vs Whisper+NLLB side-by-side on shared languages
- SOTA gap table: how our results compare to published systems (if `SOTA_FILE` is set)
- Plots: BLEU by language × direction; experiment progression

**Estimated runtime:** ~2–4 hours total (4 experiments × 30–60 min each)

---

### Paper 2 — Adaptation  (`PAPER_MODE = "adaptation"`)

**Research question:** How much does LoRA fine-tuning on in-domain data improve over
zero-shot baselines? How efficiently does performance scale with training data size?

**Setup:**
```python
PAPER_MODE        = "adaptation"
DEBUG_MODE        = False
ENABLE_FINETUNING = True           # Fine-tuning is the core contribution of this paper
FINETUNING_METHOD = "lora"         # LoRA is the default; see options below
SCALING_BUDGETS   = [100, 500, 1000, 0]  # Training sample sizes; 0 = full train set
SOTA_FILE         = ""
EXPERIMENTS       = ["FLEURS__SeamlessM4Tv2", "FLEURS__WhisperNLLB"]
```

**Fine-tuning method options:**

| `FINETUNING_METHOD` | What is updated | GPU memory needed | When to use |
|---|---|---|---|
| `"lora"` | LoRA adapter matrices only (~0.5–2% of parameters) | ~8 GB (T4 compatible) | **Default — best tradeoff** |
| `"adapter"` | Bottleneck adapter modules inserted into each layer | ~10 GB | Alternative if LoRA is unstable |
| `"full"` | All model parameters | ~24–40 GB (A100 required) | Only when you have Colab Pro + A100 |

**Data scaling budgets:**
- `SCALING_BUDGETS = [100, 500, 1000, 0]` runs 4 fine-tuning rounds at different data sizes
- `0` means the full training split (no limit)
- Produces a **learning curve** showing BLEU vs number of training samples
- Set `SCALING_BUDGETS = []` to skip scaling and only do a single full fine-tune

**Expected outputs:**
- Before/after fine-tuning comparison table (BLEU gain per language)
- Data scaling curves (BLEU as a function of training samples)
- Fine-tuning summary tables per language per method

**Estimated runtime:** ~4–6 hours (fine-tuning adds ~30–90 min per language per experiment)

---

### Paper 3 — Audio Strategies  (`PAPER_MODE = "audio"`)

**Research question:** How do different audio preprocessing choices affect translation quality?
When should you normalise, trim silence, or chunk long utterances?

**Setup:**
```python
PAPER_MODE        = "audio"
DEBUG_MODE        = False
ENABLE_FINETUNING = False          # Set True to also fine-tune each audio path
SCALING_BUDGETS   = []
SOTA_FILE         = ""
EXPERIMENTS       = ["FLEURS__SeamlessM4Tv2", "FLEURS__WhisperNLLB"]
```

**Audio strategies evaluated:**

| Strategy | Description | Hypothesis |
|---|---|---|
| Direct audio | Raw waveform, no preprocessing | Baseline |
| Normalised audio | Amplitude-normalised to target RMS level | Helps when recordings have inconsistent volume |
| Trimmed audio | Leading/trailing silence removed | Helps when silence confuses the model's attention |
| Chunk-based audio | Fixed-length segments (e.g. 10 s) with overlap | Helps for long utterances that exceed the model's context |

**Text MT ceiling:** The experiment also runs gold-transcript → English translation
(bypassing ASR entirely). This is the upper bound for any audio path.

**Expected outputs:**
- Strategy comparison table: BLEU and ChrF per strategy × language
- Text-MT ceiling vs audio BLEU (quantifies the ASR contribution to error)
- Duration distribution plots, silence-ratio histograms (EDA)
- Recommendation: which strategy works best per language

**Estimated runtime:** ~2–4 hours (strategy enumeration is the main cost)

---

### Paper 4 — Cascade vs End-to-End  (`PAPER_MODE = "cascade"`)

**Research question:** When does the cascade pipeline (Whisper ASR → NLLB MT) outperform
the end-to-end model (SeamlessM4T), and vice versa? What is the break-even ASR quality?

**Setup:**
```python
PAPER_MODE        = "cascade"
DEBUG_MODE        = False
ENABLE_FINETUNING = False
SCALING_BUDGETS   = []
SOTA_FILE         = ""
EXPERIMENTS       = ["FLEURS__SeamlessM4Tv2", "FLEURS__WhisperNLLB"]
```

**Analyses run automatically when `PAPER_MODE = "cascade"`:**

| Analysis | What it does | Output file |
|---|---|---|
| Oracle cascade | Runs NLLB MT on **gold** source transcripts (no ASR errors) | `oracle_cascade_metrics.csv` |
| Error propagation | Injects WER-controlled noise into transcripts; measures BLEU drop | `error_propagation_metrics.csv` |
| Break-even WER | Linear regression to find the ASR WER where cascade ≈ end-to-end | `breakeven_metrics.csv` |

**For latency and VRAM profiling** (run separately after the main experiments):
```bash
python papers/paper4_cascade/run_cascade_analysis.py
```
This measures median latency (ms) and peak VRAM (MB) per architecture.

**Expected outputs:**
- E2E vs cascade BLEU comparison per language
- Oracle ceiling: what cascade could achieve with perfect ASR
- Error propagation plot: BLEU vs injected WER
- Break-even WER per language (the quality threshold for choosing cascade)

**Estimated runtime:** ~2–3 hours

---

### Paper 5 — Cross-Lingual Transfer  (`PAPER_MODE = "transfer"`)

**Research question:** Does multilingual pretraining transfer better within typological
language families (Niger-Congo: Yoruba, Igbo) than across families (Afro-Asiatic: Hausa)?

**Setup:**
```python
PAPER_MODE        = "transfer"
DEBUG_MODE        = False
ENABLE_FINETUNING = False
SCALING_BUDGETS   = []
SOTA_FILE         = ""
EXPERIMENTS       = ["FLEURS__SeamlessM4Tv2", "FLEURS__WhisperNLLB"]
```

**Analyses run automatically when `PAPER_MODE = "transfer"`:**

| Analysis | What it does | Output file |
|---|---|---|
| Typological similarity | Computes URIEL syntax_knn cosine similarity between language pairs via lang2vec | `typological_similarity.csv` |
| Cross-lingual transfer | Fine-tunes on source language; evaluates on target language | `crosslingual_transfer_metrics.csv` |
| Few-shot scaling | Learning curves per language at multiple sample budgets | `few_shot_scaling_metrics.csv` |
| Interaction regression | Tests whether log(samples) × language-family interaction is significant | `interaction_regression.csv` |

> lang2vec is required for typological similarity: `pip install lang2vec`

**Expected outputs:**
- Typological distance matrix across all language pairs
- Transfer efficiency table (how much fine-tuning on language A helps language B)
- Learning curves showing data efficiency by language and family
- Statistical test results for the family-membership hypothesis

**Estimated runtime:** ~3–4 hours (transfer experiments multiply fine-tuning runs)

---

## Parameter reference — every variable explained

### `DEBUG_MODE`
Controls the sample size used for evaluation.

| Value | Text samples | Audio samples | Runtime per experiment | Use for |
|---|---|---|---|---|
| `True` | 8 | 3 | ~10 min | Checking the pipeline works; not for publication |
| `False` | 100–300 | 30–100 | ~30–60 min | Publication-quality numbers |

> **Always start with `DEBUG_MODE = True`** to verify the entire pipeline runs without errors.
> Only switch to `False` once you are confident the setup is correct.

---

### `FAST_MODE`
When set to `True` in `run_experiment.py`, it forces `DEBUG_MODE = True`.
Useful for rapid iteration. Not exposed in this notebook (set directly in `run_experiment.py` if needed).

---

### `RUN_FULL_GRID`
Controls whether all 3 experiment grid points (Experiment_1, Experiment_2, Experiment_3)
are evaluated in full mode, or only Experiment_1.

- `True` (default): runs Experiments 1, 2, and 3 — each with a different number of
  evaluation samples (100, 200, 300 for text; 30, 75, 100 for audio)
- `False`: runs only Experiment_1 — faster, less data, still valid for papers 1 and 4

Not exposed in this notebook; edit `run_experiment.py` directly if needed.

---

### `ENABLE_FINETUNING`
Whether to fine-tune the models before evaluation.

- `False`: zero-shot evaluation only (Papers 1, 3, 4, 5)
- `True`: run fine-tuning first, then evaluate before and after (Papers 2, 3)

When `True`, the script runs fine-tuning for each enabled task:
- Text translation (source→English)
- Reverse translation (English→source)
- ASR (speech→source transcript)
- Direct speech translation (SeamlessM4T only, disabled by default)

---

### `FINETUNING_METHOD`
Which fine-tuning approach to use. Only relevant when `ENABLE_FINETUNING = True`.

| Value | Method | Parameters trained | VRAM | Speed | Quality |
|---|---|---|---|---|---|
| `"lora"` | Low-Rank Adaptation | ~0.5–2% of model params | ~8 GB | Fast | Good |
| `"adapter"` | Bottleneck adapters | ~1–5% of model params | ~10 GB | Medium | Good |
| `"full"` | Full fine-tuning | 100% of model params | ~24–40 GB | Slow | Best |

**Recommendation:** Use `"lora"` on T4 (free Colab). Use `"full"` only on A100 (Colab Pro).

---

### `SCALING_BUDGETS`
A list of training sample counts for data-scaling experiments (Paper 2 only).

```python
SCALING_BUDGETS = [100, 500, 1000, 0]   # 0 = full train split (no limit)
SCALING_BUDGETS = []                     # disabled — single full fine-tune only
```

When non-empty, the script fine-tunes at each budget size and evaluates after each,
producing a learning curve. This reveals how efficiently the model learns from
limited in-domain data.

---

### `SOTA_FILE`
Path to a CSV of published baseline results for comparison in the report.

```python
SOTA_FILE = "sota/paper1_benchmark/sota_results.csv"   # relative to repo root
SOTA_FILE = ""                                          # skip SOTA comparison
```

**CSV format** (required columns: `system`, `language`, `BLEU`; optional: `ChrF`, `WER`, `direction`, `venue`, `year`):
```
system,language,BLEU,ChrF,direction,venue,year
Helsinki-NLP/opus-mt,yoruba,12.4,32.1,source_to_english,ACL,2023
mBART-50,hausa,8.1,25.3,source_to_english,EMNLP,2022
seamless-m4t-v2,swahili,18.6,41.2,source_to_english,Meta,2023
```

> Populate this file before running to get the SOTA gap section in the report.

---

### `EXPERIMENTS`
The list of pipelines to run. Comment out any to skip.

```python
EXPERIMENTS = [
    "AfricanCeltic__SeamlessM4Tv2",   # E2E, IWSLT domain, Igbo+Yoruba
    "AfricanCeltic__WhisperNLLB",     # Cascade, IWSLT domain, Yoruba+Hausa
    "FLEURS__SeamlessM4Tv2",          # E2E, FLEURS benchmark, Igbo+Yoruba+Swahili
    "FLEURS__WhisperNLLB",            # Cascade, FLEURS benchmark, Yoruba+Hausa+Swahili
]
```

**When to skip experiments:**
- Resuming after a crash: comment out already-completed experiments
- Papers 2–5: FLEURS-only results are sufficient for the main analysis; African-Celtic
  can be added for supplementary cross-domain comparison
- Paper 1 only: all 4 are needed for the full cross-experiment comparison

---

### `FORCE_RERUN`
Not exposed in this notebook (set in `run_experiment.py` directly).
When `True`, deletes and rebuilds the dataset cache even if it already exists.
Use only if you suspect a corrupted cache.

---

## Recommended workflow for a complete paper run

```
1. DEBUG smoke test (20–40 min)
   DEBUG_MODE = True, all 4 experiments, your target PAPER_MODE
   → Confirms the pipeline runs end-to-end without errors

2. Full paper run (2–6 hours, depending on paper)
   DEBUG_MODE = False, configure per the recipe for your paper above
   → Publication-quality metrics

3. Review the report
   Open papers/<paper_id>/results_report.md
   → Read observations, verify numbers, fill [NARRATIVE:...] sections

4. Iterate
   Adjust SOTA_FILE, re-run Step 7 only (no need to re-run experiments)
   → Updated report without re-running experiments
```

## Step 1 — Mount Google Drive

Mounts your Google Drive so that outputs and the results report are automatically backed up
to `MyDrive/LinguoMT-AfricaS2T/` at the end of the run.

> **Optional but recommended.** If you skip this step the run still works; outputs will only
> be available via the browser download in Step 8 and will be lost when the Colab session ends.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted.")
except ImportError:
    print("Not in Colab — Drive mount skipped.")

## Step 2 — Clone / Update Repository

Clones `prsisda/LinguoMT-AfricaS2T` into `/content/LinguoMT-AfricaS2T`.
If the directory already exists (e.g. on a resumed session) it resets to the latest `main`.

> Re-run this cell if you suspect the repo is out of date.

In [ ]:
import os, subprocess

REPO_DIR = "/content/LinguoMT-AfricaS2T"
REPO_URL = "https://github.com/prsisda/LinguoMT-AfricaS2T.git"

if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)
    print("Repo updated to origin/main")
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("Repo cloned")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## Step 3 — Install Dependencies

Installs all Python packages required by the framework and the models.

| Package | Purpose |
|---|---|
| `transformers>=4.40` | SeamlessM4T-v2 and Whisper models |
| `datasets` | FLEURS and African-Celtic dataset loading |
| `sacrebleu` | BLEU and ChrF metric computation |
| `jiwer` | WER / CER metric computation |
| `librosa`, `soundfile` | Audio loading and preprocessing |
| `sentencepiece` | NLLB and SeamlessM4T tokenisation |
| `accelerate` | Multi-GPU / mixed-precision inference |
| `torchcodec` | GPU-accelerated audio decoding (optional; falls back gracefully) |
| `tabulate` | Markdown table formatting in the report |

> This cell is safe to re-run. Installation is skipped for packages already present.

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-U",
    "transformers>=4.40", "datasets", "sacrebleu", "librosa", "soundfile",
    "sentencepiece", "accelerate", "jiwer", "pandas==2.2.2",
    "pyarrow>=15.0.0", "protobuf", "tabulate",
], check=True)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "torchcodec",
    "--extra-index-url", "https://download.pytorch.org/whl/cu121"], check=False)
print("\nDependencies ready.")

## Step 4 — Cache & Pipeline Setup

**Run once, then leave it.** These settings control where models and datasets
are stored and how many sample pairs are pre-built. You do **not** need to
re-run this step when changing paper mode or eval parameters.

| Variable | Default | Purpose |
|---|---|---|
| `EXPERIMENTS` | all 4 | Which experiment pipelines to include |
| `HF_CACHE_DIR` | Drive path | Where model weights are stored (persist across sessions) |
| `DATASET_CACHE_DIR` | Drive path | Where pre-built sample pairs are stored |
| `MAX_CACHED_PAIRS` | 300 | Max text pairs per language cached in Step 6; must be ≥ `max(EVAL_TEXT_SAMPLES)` |

> **When to re-run each step**
>
> | Step | Re-run when… |
> |---|---|
> | 4 | You change `EXPERIMENTS`, cache paths, or `MAX_CACHED_PAIRS` |
> | 5 | New models added to `EXPERIMENTS` |
> | 6 | `MAX_CACHED_PAIRS` changed, or a new experiment added |
> | 7 | Every time you want to change paper mode or eval settings → then re-run 8–11 |
> | 8–11 | After every Step 7 change |


In [ ]:
import re, pathlib

# ── SELECT EXPERIMENTS (comment out any to skip) ──────────────────────
EXPERIMENTS = [
    "AfricanCeltic__SeamlessM4Tv2",
    "AfricanCeltic__WhisperNLLB",
    "FLEURS__SeamlessM4Tv2",
    "FLEURS__WhisperNLLB",
]
# ─────────────────────────────────────────────────────────────────────

# ── HUGGINGFACE CACHE ─────────────────────────────────────────────────
# When Drive is mounted (Step 1), set this to a Drive path so all model
# and dataset downloads persist across Colab sessions — pay once, reuse
# forever. Step 5 will create this directory and populate it.
# Set to "" to use Colab's ephemeral cache (/root/.cache/huggingface).
HF_CACHE_DIR = "/content/drive/MyDrive/LinguoMT-AfricaS2T/hf_cache"
# ─────────────────────────────────────────────────────────────────────

# ── DATASET SAMPLE CACHE ──────────────────────────────────────────────
# Step 6 pre-processes datasets and saves aligned sample pairs here.
# Set to a Drive path so the prepared pairs persist across sessions.
# Set to "" to use a local session-only cache (lost when session ends).
DATASET_CACHE_DIR = "/content/drive/MyDrive/LinguoMT-AfricaS2T/dataset_cache"
# ─────────────────────────────────────────────────────────────────────

# ── MAX PAIRS TO PRE-CACHE ────────────────────────────────────────────
# Step 6 caches this many text pairs per language per experiment.
# Must be >= max(EVAL_TEXT_SAMPLES) set in Step 7.
# If you increase EVAL_TEXT_SAMPLES beyond this value, also increase
# MAX_CACHED_PAIRS here and re-run Step 6 to rebuild the cache.
MAX_CACHED_PAIRS = 300
# ─────────────────────────────────────────────────────────────────────

SCRIPTS = {exp: pathlib.Path(f"{exp}/notebooks/run_experiment.py") for exp in EXPERIMENTS}

print(f"Experiments      : {', '.join(EXPERIMENTS)}")
print(f"HF cache         : {HF_CACHE_DIR or '(ephemeral)'}")
print(f"Data cache       : {DATASET_CACHE_DIR or '(session-only)'}")
print(f"Max cached pairs : {MAX_CACHED_PAIRS}")


## Step 5 — Pre-download & Cache Models/Datasets

Downloads all models and datasets required by the selected experiments into
the `HF_CACHE_DIR` path configured in Step 4.

**Run this step once** per Colab session (or once ever if `HF_CACHE_DIR` points to
Google Drive). On all subsequent runs — including re-runs with different parameter
settings in Step 4 — models and datasets load instantly from cache with no wait.

### What gets downloaded

| Trigger | What | Approx. size |
|---|---|---|
| Any `SeamlessM4T` experiment selected | `facebook/seamless-m4t-v2-large` | ~9 GB |
| Any `WhisperNLLB` experiment selected | `openai/whisper-large-v3` | ~3 GB |
| Any `WhisperNLLB` experiment selected | `facebook/nllb-200-distilled-600M` | ~2.5 GB |
| Any `AfricanCeltic` experiment selected | `McGill-NLP/african_celtic_dataset` (`dev` split) | ~2 GB |
| Any `FLEURS` experiment selected | `google/fleurs` (per-language configs) | ~0.5 GB per language |

> Only evaluation splits are cached here (`dev` / `validation`).
> Training splits are downloaded on-demand the first time `ENABLE_FINETUNING = True`.

### Why set `HF_CACHE_DIR` to Drive

Colab's default cache (`/root/.cache/huggingface`) is **ephemeral** — it is erased when
the session ends. Pointing `HF_CACHE_DIR` at a Drive path means ~15 GB of weights and
dataset shards are written once and reused across all future sessions automatically.


In [ ]:
import os, subprocess, sys
from pathlib import Path

# ── 1. Point HuggingFace to the cache directory ───────────────────────────────
if HF_CACHE_DIR:
    Path(HF_CACHE_DIR).mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = HF_CACHE_DIR
    print(f"HF cache : {HF_CACHE_DIR}  (Drive-backed — persists across sessions)")
else:
    print(f"HF cache : {os.environ.get('HF_HOME', '/root/.cache/huggingface')}  (ephemeral)")

# ── 2. Determine what the selected experiments need ───────────────────────────
_NEEDS_SEAMLESS = any("SeamlessM4Tv2" in e for e in EXPERIMENTS)
_NEEDS_WHISPER  = any("WhisperNLLB"   in e for e in EXPERIMENTS)
_NEEDS_AC       = any("AfricanCeltic" in e for e in EXPERIMENTS)
_NEEDS_FLEURS   = any("FLEURS"        in e for e in EXPERIMENTS)

_FLEURS_CONFIGS = sorted({
    cfg
    for exp, cfgs in {
        "FLEURS__SeamlessM4Tv2": ["ig_ng", "yo_ng", "sw_ke"],
        "FLEURS__WhisperNLLB":   ["yo_ng", "ha_ng", "sw_ke"],
    }.items()
    for cfg in cfgs
    if exp in EXPERIMENTS
})

print(f"\nNeed Seamless    : {_NEEDS_SEAMLESS}")
print(f"Need Whisper+NLLB: {_NEEDS_WHISPER}")
print(f"Need AC dataset  : {_NEEDS_AC}")
print(f"Need FLEURS      : {_NEEDS_FLEURS}  configs={_FLEURS_CONFIGS}")

# ── 3. Cache models (snapshot_download — files only, no RAM loading) ──────────
# This works on CPU and GPU equally: it copies weights to disk and exits.
# The actual model is only loaded into RAM when an experiment runs in Step 6.
from huggingface_hub import snapshot_download

def _cache_model(label, repo_id):
    print(f"\n  [{label}]  downloading/verifying...", end=" ", flush=True)
    try:
        snapshot_download(repo_id=repo_id, repo_type="model")
        print("cached.")
    except Exception as exc:
        print(f"WARN: {exc}")

if _NEEDS_SEAMLESS:
    _cache_model("SeamlessM4T-v2-Large (~9 GB)", "facebook/seamless-m4t-v2-large")

if _NEEDS_WHISPER:
    _cache_model("Whisper-large-v3 (~3 GB)",     "openai/whisper-large-v3")
    _cache_model("NLLB-200-600M (~2.5 GB)",      "facebook/nllb-200-distilled-600M")

# ── 4. Cache datasets (evaluation splits only; training splits load on demand) ─
from datasets import load_dataset

def _cache_dataset(ds_id, cfg, split):
    label = f"{ds_id} [{cfg}] split={split}"
    print(f"\n  [{label}]  downloading/verifying...", end=" ", flush=True)
    try:
        ds = load_dataset(ds_id, cfg, split=split, trust_remote_code=True)
        n  = len(ds) if hasattr(ds, "__len__") else "?"
        del ds
        print(f"cached  ({n} rows).")
    except Exception as exc:
        print(f"WARN: {exc}")

if _NEEDS_AC:
    _cache_dataset("McGill-NLP/african_celtic_dataset", "default", "dev")

for _cfg in _FLEURS_CONFIGS:
    _cache_dataset("google/fleurs", _cfg, "validation")

# ── 5. Report cache size ──────────────────────────────────────────────────────
_cache_loc = os.environ.get("HF_HOME", "/root/.cache/huggingface")
try:
    _sz = subprocess.run(["du", "-sh", _cache_loc], capture_output=True, text=True).stdout.split()
    print(f"\nCache at {_cache_loc} — total size: {_sz[0] if _sz else 'unknown'}")
except Exception:
    pass
print("\nPre-download complete. Switch to a GPU runtime to run experiments (Steps 6-9).")


## Step 6 — Prepare & Cache Data Samples

Scans the downloaded datasets, aligns source text with English translations, and
saves the selected sample pairs as `.pkl` files to `DATASET_CACHE_DIR` on Drive.

**Run once per configuration.** Re-run only if you change `EXPERIMENTS`, languages,
`EVAL_TEXT_SAMPLES`, `EVAL_AUDIO_SAMPLES`, or `DEBUG_MODE` in Step 4.
Changing `PAPER_MODE`, `ENABLE_FINETUNING`, or `SOTA_FILE` does **not** require re-running.

### What this step produces

For each selected experiment × language combination, a `.pkl` file is saved at
`DATASET_CACHE_DIR/<experiment>/pairs_<adapter>_<hash>.pkl` containing:
- Up to `max(EVAL_TEXT_SAMPLES)` aligned `(src_text, eng_text, src_audio, eng_audio)` tuples
- Hash encodes dataset, split, languages, and sample count — experiments look up the exact file

### Time

| State | Time |
|---|---|
| Cold — first run, files not on Drive | ~5–15 min (streams and aligns dataset) |
| Warm — files already on Drive | < 5 sec (loads `.pkl` directly) |

> The `.pkl` files contain only text metadata, not audio bytes — they are small (~1 MB each).
> Audio is decoded on-demand during inference in Step 7.


In [ ]:
import sys, os
from pathlib import Path

REPO_DIR = "/content/LinguoMT-AfricaS2T"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

from framework.languages import AFRICAN_LANGUAGES, get_adapter_type
from framework.dataset import DatasetCache

# ── Per-experiment definitions (mirrors MANUAL_LANGUAGES in each script) ──────
_EXP_DEFS = {
    "AfricanCeltic__SeamlessM4Tv2": {
        "dataset_id": "McGill-NLP/african_celtic_dataset",
        "split":      "dev",
        "languages":  ["igbo", "yoruba"],
    },
    "AfricanCeltic__WhisperNLLB": {
        "dataset_id": "McGill-NLP/african_celtic_dataset",
        "split":      "dev",
        "languages":  ["yoruba", "hausa"],
    },
    "FLEURS__SeamlessM4Tv2": {
        "dataset_id": "google/fleurs",
        "split":      "validation",
        "languages":  ["igbo", "yoruba", "swahili", "amharic", "zulu", "wolof"],
    },
    "FLEURS__WhisperNLLB": {
        "dataset_id": "google/fleurs",
        "split":      "validation",
        "languages":  ["yoruba", "hausa", "swahili"],
    },
}

# MAX_CACHED_PAIRS comes from Step 4 — always cache a generous upper bound
# so experiments can use any EVAL_TEXT_SAMPLES <= MAX_CACHED_PAIRS without
# needing to rebuild the cache.
_max_pairs = MAX_CACHED_PAIRS
_max_scan  = 50000
print(f"max_pairs={_max_pairs}  max_scan_rows={_max_scan}\n")

# ── Build cache for each selected experiment ──────────────────────────────────
for exp_name in EXPERIMENTS:
    defn = _EXP_DEFS[exp_name]

    lang_cfgs = [
        {"language_key": lk, **AFRICAN_LANGUAGES[lk]}
        for lk in defn["languages"]
        if lk in AFRICAN_LANGUAGES
    ]

    cache_dir = (
        Path(DATASET_CACHE_DIR) / exp_name
        if DATASET_CACHE_DIR
        else Path(REPO_DIR) / exp_name / "cache"
    )

    print(f"{'='*64}")
    print(f"  Experiment : {exp_name}")
    print(f"  Dataset    : {defn['dataset_id']}  split={defn['split']}")
    print(f"  Languages  : {defn['languages']}")
    print(f"  Cache dir  : {cache_dir}")
    print(f"{'='*64}")

    dc = DatasetCache(
        dataset_id=defn["dataset_id"],
        adapter_type=get_adapter_type(defn["dataset_id"]),
        language_configs=lang_cfgs,
        split=defn["split"],
        max_pairs=_max_pairs,
        max_scan_rows=_max_scan,
        cache_dir=cache_dir,
        force_rerun=False,
    )
    dc.build()
    stats = dc.stats()
    print(f"  Pairs ready: {stats}")
    for lk, n in stats.items():
        if n == 0:
            print(f"  WARNING: {lk} returned 0 pairs — check dataset or language config")
    print()

print("Data preparation complete. Experiments will skip dataset scanning.")


## Step 7 — Configure Experiment Settings

**Edit this cell before every paper run.** Steps 5 and 6 (model/data cache) are
already done — you only need to change settings here and re-run Steps 8–11.

| Variable | Options | Purpose |
|---|---|---|
| `PAPER_MODE` | benchmark / adaptation / audio / cascade / transfer | Which paper's experiment to run |
| `DEBUG_MODE` | True / False | True = tiny sample for quick testing; False = full paper run |
| `N_EVAL_RUNS` | 1–3 | How many evaluation passes (each on a larger sample) |
| `EVAL_TEXT_SAMPLES` | list of ints ≤ `MAX_CACHED_PAIRS` | Text pairs per pass |
| `EVAL_AUDIO_SAMPLES` | list of ints | Audio utterances per pass |
| `SCALING_BUDGETS` | list of ints, or `[]` | Paper 2 only — training set sizes for learning curve |


In [ ]:
import pathlib

# ── SELECT PAPER (uncomment exactly one) ──────────────────────────────
PAPER_MODE = "benchmark"    # Paper 1 — zero-shot baselines           ← ACTIVE
# PAPER_MODE = "adaptation" # Paper 2 — fine-tuning comparison
# PAPER_MODE = "audio"      # Paper 3 — audio strategy analysis
# PAPER_MODE = "cascade"    # Paper 4 — cascade vs end-to-end
# PAPER_MODE = "transfer"   # Paper 5 — cross-lingual transfer
# ─────────────────────────────────────────────────────────────────────

# ── RUN MODE ──────────────────────────────────────────────────────────
DEBUG_MODE = True   # True ≈ 40 min total | False ≈ 2–4 hours total
# ─────────────────────────────────────────────────────────────────────

# ── FINE-TUNING (Papers 2 & 3 only) ──────────────────────────────────
ENABLE_FINETUNING = False
FINETUNING_METHOD = "lora"   # lora | adapter | full
# ─────────────────────────────────────────────────────────────────────

# ── EVALUATION PASSES ────────────────────────────────────────────────
# How many times to evaluate, each on a larger sample.
#   1 → one result per language (fastest; Experiment_1 only)
#   2 → two results per language (Experiment_1 + Experiment_2)
#   3 → three results per language (default; recommended for papers)
N_EVAL_RUNS = 3
# Text pairs per pass — must be <= MAX_CACHED_PAIRS (set in Step 4).
EVAL_TEXT_SAMPLES  = [100, 200, 300]   # pass 1, pass 2, pass 3
# Audio utterances per pass. Audio is slower than text.
EVAL_AUDIO_SAMPLES = [ 30,  75, 100]   # pass 1, pass 2, pass 3
# ─────────────────────────────────────────────────────────────────────

# ── DATA SCALING BUDGETS (Paper 2 only) ──────────────────────────────
SCALING_BUDGETS = []
# ─────────────────────────────────────────────────────────────────────

# ── SOTA FILE (optional) ──────────────────────────────────────────────
SOTA_FILE = ""
# ─────────────────────────────────────────────────────────────────────

# ── FORCE RERUN ───────────────────────────────────────────────────────
FORCE_RERUN = False
# ─────────────────────────────────────────────────────────────────────

if max(EVAL_TEXT_SAMPLES or [0]) > MAX_CACHED_PAIRS:
    print(f"WARNING: max(EVAL_TEXT_SAMPLES)={max(EVAL_TEXT_SAMPLES)} > MAX_CACHED_PAIRS={MAX_CACHED_PAIRS}")
    print("         Increase MAX_CACHED_PAIRS in Step 4 and re-run Step 6 first.")

print(f"Paper      : {PAPER_MODE}")
print(f"Mode       : {'DEBUG  (fast test)' if DEBUG_MODE else 'FULL   (paper run)'}")
print(f"Fine-tune  : {ENABLE_FINETUNING}  (method: {FINETUNING_METHOD})")
print(f"Eval runs  : {N_EVAL_RUNS}  text={EVAL_TEXT_SAMPLES}  audio={EVAL_AUDIO_SAMPLES}")
print(f"Scaling    : {SCALING_BUDGETS if SCALING_BUDGETS else 'disabled'}")
print(f"SOTA file  : {SOTA_FILE or 'disabled'}")


## Step 8 — Patch Scripts & Run All Experiments

**Sub-step A** writes your Step 7 settings into each `run_experiment.py` file
(DEBUG_MODE, PAPER_MODE, ENABLE_FINETUNING, HF_CACHE_DIR, DATASET_CACHE_DIR, etc.)
and resets the repo to the latest `origin/main`.

**Sub-step B** runs each selected experiment sequentially.
Models load from the HF cache (Step 5). Sample pairs load from the data cache (Step 6).
No downloading or dataset scanning during experiments.

Each experiment outputs to `/content/outputs/<timestamp_slug>/`:

```
<timestamp_slug>/
├── config.json                  — full configuration snapshot
├── metrics/
│   ├── text_metrics.csv         — BLEU, ChrF per language and direction
│   ├── audio_metrics.csv        — BLEU, ChrF per audio strategy, language, direction
│   └── asr_metrics.csv          — WER, CER per language
├── tables/                      — Markdown and CSV summary tables
├── plots/                       — Score charts and EDA visualisations (PNG)
├── interpretations/             — Auto-generated interpretation fragments
├── summaries/                   — Per-experiment summary Markdown
└── predictions/qualitative/     — Side-by-side source / reference / hypothesis examples
```

> If one experiment fails, the others will not run (the cell raises a `RuntimeError`).
> Fix the issue and re-run this cell. To skip a failing experiment, comment it out in Step 4.


In [ ]:
import subprocess, re

subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "reset", "--hard", "origin/main"], check=True)
print("Repository updated.")

mode_str            = "True" if DEBUG_MODE else "False"
ft_str              = "True" if ENABLE_FINETUNING else "False"
force_rerun_str     = "True" if FORCE_RERUN else "False"
budgets_str         = repr(SCALING_BUDGETS)
hf_cache_str        = repr(HF_CACHE_DIR)
dataset_cache_str   = repr(DATASET_CACHE_DIR)
max_cached_pairs_str = str(MAX_CACHED_PAIRS)

for name, sp in SCRIPTS.items():
    src = sp.read_text()
    src = re.sub(r"(?m)^(DEBUG_MODE\s*=\s*)(True|False)",             rf"\g<1>{mode_str}",           src)
    src = re.sub(r'(?m)^(PAPER_MODE\s*=\s*)["\'][^"\']+["\']',        rf'\g<1>"{PAPER_MODE}"',        src)
    src = re.sub(r"(?m)^(ENABLE_FINETUNING\s*=\s*)(True|False)",      rf"\g<1>{ft_str}",              src)
    src = re.sub(r"(?m)^(FINETUNING_METHOD\s*=\s*)['\"][^'\"]+['\"]", rf"\g<1>'{FINETUNING_METHOD}'", src)
    src = re.sub(r"(?m)^(SCALING_BUDGETS\s*=\s*)\[[^\]]*\]",          rf"\g<1>{budgets_str}",         src)
    src = re.sub(r"(?m)^(N_EVAL_RUNS\s*=\s*)\d+",                     rf"\g<1>{N_EVAL_RUNS}",         src)
    src = re.sub(r"(?m)^(EVAL_TEXT_SAMPLES\s*=\s*)\[[^\]]*\]",        rf"\g<1>{repr(EVAL_TEXT_SAMPLES)}",  src)
    src = re.sub(r"(?m)^(EVAL_AUDIO_SAMPLES\s*=\s*)\[[^\]]*\]",       rf"\g<1>{repr(EVAL_AUDIO_SAMPLES)}", src)
    src = re.sub(r'(?m)^(SOTA_FILE\s*=\s*)["\'][^"\']*["\']',         rf'\g<1>"{SOTA_FILE}"',         src)
    src = re.sub(r'(?m)^(HF_CACHE_DIR\s*=\s*)["\'][^"\']*["\']',      rf'\g<1>{hf_cache_str}',        src)
    src = re.sub(r'(?m)^(DATASET_CACHE_DIR\s*=\s*)["\'][^"\']*["\']', rf'\g<1>{dataset_cache_str}',   src)
    src = re.sub(r'(?m)^(MAX_CACHED_PAIRS\s*=\s*)\d+',         rf'\g<1>{max_cached_pairs_str}', src)
    src = re.sub(r'(?m)^(FORCE_RERUN\s*=\s*)(True|False)',       rf'\g<1>{force_rerun_str}',      src)
    sp.write_text(src)
    print(f"  Patched: {name}")

print(f"\nScripts configured for Step 8 — {PAPER_MODE} | {'DEBUG' if DEBUG_MODE else 'FULL'}")


In [ ]:
import sys, subprocess as _sp, os as _os

_env = {**_os.environ}
if HF_CACHE_DIR:
    _env["HF_HOME"] = HF_CACHE_DIR

for name, sp in SCRIPTS.items():
    print(f"\n{'='*64}\n  Running : {name}\n  Paper   : {PAPER_MODE}  |  Mode : {'DEBUG' if DEBUG_MODE else 'FULL'}\n{'='*64}\n")
    result = _sp.run([sys.executable, str(sp)], env=_env)
    if result.returncode != 0:
        raise RuntimeError(f"{name} failed (exit code {result.returncode})")

print("\nAll experiments finished.")


## Step 9 — Consolidate Metrics

Scans all `/content/outputs/*/metrics/` directories and merges the metric CSVs
into a single consolidated folder:

```
/content/outputs/consolidated_<timestamp>/
├── text_metrics_all.csv    — all text MT results (BLEU, ChrF) in one file
├── asr_metrics_all.csv     — all ASR results (WER, CER) in one file
├── audio_metrics_all.csv   — all audio strategy results in one file
└── metrics_summary.md      — Markdown overview table for quick inspection
```

An `experiment` column identifies which pipeline each row came from.
The consolidated directory is what the report generator (Step 9) reads.


In [ ]:
import json as _json
import pandas as pd
from pathlib import Path
from datetime import datetime

out_root         = Path("/content/outputs")
consolidated_dir = out_root / f"consolidated_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"
consolidated_dir.mkdir(parents=True, exist_ok=True)

all_text, all_asr, all_audio = [], [], []

for run_dir in sorted(out_root.glob("*/")):
    if "consolidated" in run_dir.name:
        continue
    exp_label = run_dir.name
    cfg_path  = run_dir / "config.json"
    if cfg_path.exists():
        exp_label = _json.loads(cfg_path.read_text()).get("experiment_family", run_dir.name)
    for fname, store in [
        ("text_metrics.csv",  all_text),
        ("asr_metrics.csv",   all_asr),
        ("audio_metrics.csv", all_audio),
    ]:
        fpath = run_dir / "metrics" / fname
        if fpath.exists():
            df = pd.read_csv(fpath)
            if "experiment" not in df.columns:
                df.insert(0, "experiment", exp_label)
            store.append(df)

summary_parts = [f"# LinguoMT Consolidated Metrics — {PAPER_MODE}\n\n"]
for label, store, out_name in [
    ("Text Translation", all_text,  "text_metrics_all.csv"),
    ("ASR",              all_asr,   "asr_metrics_all.csv"),
    ("Audio",            all_audio, "audio_metrics_all.csv"),
]:
    if store:
        merged = pd.concat(store, ignore_index=True)
        merged.to_csv(consolidated_dir / out_name, index=False)
        summary_parts += [f"## {label}\n\n", merged.to_markdown(index=False), "\n\n"]
        print(f"=== {label} ===\n{merged.to_string(index=False)}\n")

(consolidated_dir / "metrics_summary.md").write_text("".join(summary_parts))
print(f"Consolidated → {consolidated_dir}")

## Step 10 — Generate Results Report

Runs `papers/generate_report.py` to produce `papers/<paper_id>/results_report.md`.

### What the report contains

| Section | Description |
|---|---|
| **Header** | Run metadata: date, experiments, languages, SOTA file used |
| **Experiment overview** | Language coverage matrix with per-experiment availability notes |
| **Text translation results** | BLEU and ChrF per experiment × language × direction; best value bolded |
| **ASR results** | WER and CER per experiment × language; best (lowest) value bolded |
| **Audio strategy results** | BLEU and ChrF per strategy × language (if audio strategies were run) |
| **Cross-experiment comparison** | SeamlessM4T vs Whisper+NLLB side-by-side, Source→English BLEU |
| **SOTA comparison** | Gap vs published systems per language (only if `SOTA_FILE` was set) |
| **Key observations** | Paper-mode-specific discussion points derived from the metrics |
| **Narrative placeholders** | `[NARRATIVE:...]` sections to fill when writing the paper |
| **Appendix** | Full raw metric tables for reference |

### Important disclaimer

> The report is auto-generated scaffolding — it gives **direction for writing**, not finished text.
> Before citing any number or claim you must:
> - Verify metrics against the raw output files in `/content/outputs/*/metrics/`
> - Conduct a systematic review of the relevant literature (not just the SOTA CSV provided)
> - Fill all `[NARRATIVE:...]` sections through your own analysis and expert judgement
> - Critically assess the discussion observations — they are heuristics, not conclusions


In [ ]:
import subprocess, sys
from pathlib import Path
from pathlib import Path

out_root = Path("/content/outputs")
cons = sorted(out_root.glob("consolidated_*/"), reverse=True)
if not cons:
    print("ERROR: No consolidated directory found — run Step 6 first.")
else:
    consolidated_dir = cons[0]
    cmd = [
        sys.executable, "papers/generate_report.py", PAPER_MODE,
        "--consolidated-dir", str(consolidated_dir),
    ]
    if SOTA_FILE:
        cmd += ["--sota-file", SOTA_FILE]

    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
    else:
        paper_id_map = {
            "benchmark":  "paper1_benchmark",
            "adaptation": "paper2_adaptation",
            "audio":      "paper3_audio",
            "cascade":    "paper4_cascade",
            "transfer":   "paper5_transfer",
        }
        report_path = Path(f"papers/{paper_id_map[PAPER_MODE]}/results_report.md")
        if report_path.exists():
            text    = report_path.read_text()
            preview = text[:5000]
            print("\n" + "="*64)
            print(f"REPORT PREVIEW  —  {report_path}")
            print("="*64)
            print(preview)
            if len(text) > 5000:
                print(f"\n... ({len(text) - 5000} more chars — open the file for the complete report)")

## Step 11 — Package & Download

Assembles a timestamped ZIP containing everything needed for paper writing:

```
linguomt_<paper_id>_<full|debug>_<timestamp>.zip
├── results_report.md               — complete analysis report
├── consolidated_metrics/           — merged CSVs and metrics_summary.md
│   ├── text_metrics_all.csv
│   ├── asr_metrics_all.csv
│   └── audio_metrics_all.csv
├── <experiment_family>/
│   ├── tables/                     — per-experiment summary tables
│   ├── plots/                      — score and EDA charts
│   ├── interpretations/            — interpretation fragments
│   └── summaries/                  — per-language summaries
└── ...
```

The ZIP is:
1. **Saved to Google Drive** at `MyDrive/LinguoMT-AfricaS2T/<filename>.zip` (if Drive is mounted)
2. **Triggered as a browser download** in Colab

> If the download does not start automatically, find the ZIP in the Colab file browser
> at `/content/<filename>.zip` and download it manually.


In [ ]:
import shutil, json as _json
from pathlib import Path
from datetime import datetime

try:
    from google.colab import files as _colab_files
    _in_colab = True
except ImportError:
    _in_colab = False

ts        = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
repo_root = Path("/content/LinguoMT-AfricaS2T")
mode_tag  = "debug" if DEBUG_MODE else "full"
paper_id_map = {
    "benchmark":  "paper1_benchmark",
    "adaptation": "paper2_adaptation",
    "audio":      "paper3_audio",
    "cascade":    "paper4_cascade",
    "transfer":   "paper5_transfer",
}
PAPER_ID = paper_id_map[PAPER_MODE]
pkg_name  = f"linguomt_{PAPER_ID}_{mode_tag}_{ts}"
pkg_dir   = Path("/content") / pkg_name
pkg_dir.mkdir(parents=True, exist_ok=True)

# 1. Results report (Markdown)
report_md = repo_root / "papers" / PAPER_ID / "results_report.md"
if report_md.exists():
    shutil.copy2(str(report_md), str(pkg_dir / "results_report.md"))

# 2. Consolidated metrics
out_root = Path("/content/outputs")
cons = sorted(out_root.glob("consolidated_*/"), reverse=True)
if cons:
    shutil.copytree(str(cons[0]), str(pkg_dir / "consolidated_metrics"))

# 3. Per-experiment tables, plots, interpretations, summaries
for run_dir in sorted(out_root.glob("*/")):
    if "consolidated" in run_dir.name:
        continue
    cfg_path = run_dir / "config.json"
    label = run_dir.name
    if cfg_path.exists():
        label = _json.loads(cfg_path.read_text()).get("experiment_family", label)
    for sub in ["tables", "plots", "interpretations", "summaries"]:
        src = run_dir / sub
        if src.exists():
            shutil.copytree(str(src), str(pkg_dir / label / sub), dirs_exist_ok=True)

zip_path = shutil.make_archive(f"/content/{pkg_name}", "zip", root_dir=str(pkg_dir))
print(f"Package: {zip_path}")

drive_dir = Path("/content/drive/MyDrive/LinguoMT-AfricaS2T")
if drive_dir.exists():
    drive_dest = drive_dir / f"{pkg_name}.zip"
    shutil.copy2(zip_path, str(drive_dest))
    print(f"Drive backup: {drive_dest}")

if _in_colab:
    _colab_files.download(zip_path)
    print("Download triggered.")
else:
    print(f"Local package: {zip_path}")